# 01. Data Validation

Validate the malaria incidence dataset, confirm it has the required schema, and check basic data quality before analysis.


In [ ]:
# !pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn joblib

import os
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

print("MSF-HIE Malaria Forecasting Pipeline")
print("=" * 60)

data_path = os.environ.get("DATA_PATH", "../data/sample")
csv_candidates = []
for dirpath, _, filenames in os.walk(data_path):
    for name in filenames:
        if name.lower().endswith((".csv", ".xlsx", ".xls")):
            csv_candidates.append(os.path.join(dirpath, name))

if csv_candidates:
    file_name = csv_candidates[0]
else:
    file_name = None

if file_name is None:
    raise FileNotFoundError("No dataset found in ../data/sample. Add a CSV or Excel file first.")

if file_name.lower().endswith(".csv"):
    df = pd.read_csv(file_name)
elif file_name.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(file_name)

print(f"Loaded file: {file_name}")
print(f"Dataset shape: {df.shape}")

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
df.head()

print("\nDataset shape:")
print(df.shape)

print("\nMissing values:")
df.isnull().sum().sort_values(ascending=False)

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nCounties:")
print(df["county"].nunique())
print(sorted(df["county"].dropna().unique()))


In [ ]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["county"] = df["county"].astype(str).str.strip()
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df = df.sort_values(["county", "date"]).reset_index(drop=True)

print("Date range:")
print(df["date"].min(), "→", df["date"].max())

required_columns = [
    "county", "date", "temperature", "precipitation", "NDVI", "ITN_coverage",
    "elevation", "population_density", "incidence_per_1000"
]

missing_required = [col for col in required_columns if col not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

print("\nAll required columns are present.")

print("\nInvalid dates:")
print(df["date"].isna().sum())

print("Negative incidence values:", (df["incidence_per_1000"] < 0).sum())
print("Negative precipitation values:", (df["precipitation"] < 0).sum())

duplicates = df.groupby(["county", "date"]).size().reset_index(name="count")
duplicates = duplicates[duplicates["count"] > 1]
print("Duplicate county-month records:", len(duplicates))
